# Tableau → Fabric: Semantic Model Generator — Play 4

> **Part of the Tableau + Microsoft Fabric AI Bridge project.**

This notebook is the **semantic model generation and deployment** stage of the pipeline.
It reads the datasource and field metadata produced by Play 2, generates a TMDL-format
semantic model definition for each datasource pointing at the Delta tables written by
Play 3, and deploys each model directly to the Fabric workspace via the Fabric REST API.

**Pipeline order:** Play 2 → Play 3 → Play 4

```
Metadata_Lakehouse (Play 2 output)
  tableau_datasources  ← model names and descriptions
  tableau_fields       ← columns, data types, roles, calculated fields
  tableau_lineage      ← upstream table names
        +
h1_ultrastore (Play 3 output)
  {datasource}_{table}  ← Delta tables (DirectLake target)
        ↓
Play 4 (this notebook)
  For each datasource:
    Generate TMDL definition
    Deploy via Fabric REST API
        ↓
Fabric Workspace
  One semantic model per Tableau datasource
  DirectLake → h1_ultrastore Delta tables
```

**What each generated semantic model contains:**
- One table per upstream source table (from Play 2 lineage)
- All columns, with data types read from the **actual landed Delta schema** (not Tableau metadata) so DirectLake binds correctly
- summarizeBy and hidden flags carried over from Play 2 metadata
- A `_Measures` calculated table with DAX stubs for all Tableau calculated fields
- DirectLake connection to h1_ultrastore, with an automatic post-deploy refresh so the model is immediately query-ready

**What it does NOT generate (by design):**
- DAX translations — calculated fields are stubbed with the original Tableau formula

**Relationships:** cross-table relationships are inferred automatically from Tableau's
hidden join keys (the disambiguated `<Base> (<Table>)` fields), with many→one direction
determined from the actual landed data, and emitted as `definition/relationships.tmdl`.

---

**Prerequisites**
- Play 2 has been run and Metadata_Lakehouse tables are current
- Play 3 has been run and h1_ultrastore Delta tables are current
- Fabric workspace managed identity has permission to create semantic models
- Fabric admin has enabled 'Service principals can use Fabric APIs' tenant setting

**Cells in this notebook**
1. Configuration
2. Load Play 2 metadata
3. TMDL generators
4. Main loop — generate and deploy semantic models
5. Verification


## ⚠️ Start Here — Plug In Your Variables

| Variable | What it is | Where to find it |
|----------|-----------|------------------|
| `WORKSPACE_ID` | Fabric workspace GUID | Fabric workspace URL |
| `DATA_LAKEHOUSE_ID` | h1_ultrastore lakehouse GUID | Fabric REST API or lakehouse settings |
| `DATA_LAKEHOUSE_NAME` | h1_ultrastore display name | e.g. `h1_ultrastore` |
| `METADATA_LAKEHOUSE` | Play 2 metadata lakehouse name | e.g. `Metadata_Lakehouse` |
| `DATASOURCE_FILTER` | Optional list of datasource names | Leave empty `[]` to process all |
| `OVERWRITE` | Whether to overwrite existing models | `True` to update, `False` to skip existing |


## Cell 1 — Configuration

Set your Fabric workspace details here. The managed identity token is obtained
automatically — no credentials needed beyond the workspace and lakehouse GUIDs.

> 🔄 **Adapting for your environment:** Update `WORKSPACE_ID`, `DATA_LAKEHOUSE_ID`,
> and `DATA_LAKEHOUSE_NAME`. Everything else is derived automatically.

In [5]:
# ── LAKEHOUSE NAMES ───────────────────────────────────────────────────────────
DATA_LAKEHOUSE_NAME = "h1_ultrastore"      # Display name of your data lakehouse
METADATA_LAKEHOUSE  = "Metadata_Lakehouse" # Display name of your metadata lakehouse

# ── FILTER CONTROLS ──────────────────────────────────────────────────────────
DATASOURCE_FILTER   = []   # e.g. ["Superstore Datasource"] — empty = process all
OVERWRITE           = True # True = update existing models, False = skip existing

# ── FABRIC REST API ──────────────────────────────────────────────────────────
FABRIC_API = "https://api.fabric.microsoft.com/v1"

import requests
import json
import base64
import uuid
import re
import time
from datetime import datetime
from pyspark.sql.types import NullType

# Get Fabric token via managed identity
token = notebookutils.credentials.getToken("pbi")
HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Get workspace ID dynamically from notebook context
WORKSPACE_ID = notebookutils.runtime.context.get("currentWorkspaceId")

# Get data lakehouse ID by display name
lakehouses_resp = requests.get(
    f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/lakehouses",
    headers=HEADERS
)
lakehouses_resp.raise_for_status()
lakehouses = lakehouses_resp.json().get("value", [])
DATA_LAKEHOUSE_ID = next(
    (l["id"] for l in lakehouses if l["displayName"] == DATA_LAKEHOUSE_NAME), None
)
if not DATA_LAKEHOUSE_ID:
    raise ValueError(f"Lakehouse '{DATA_LAKEHOUSE_NAME}' not found in workspace")

# Derive DirectLake connection URL
DIRECTLAKE_URL = f"https://onelake.dfs.fabric.microsoft.com/{WORKSPACE_ID}/{DATA_LAKEHOUSE_ID}"
EXPRESSION_SOURCE_NAME = f"DirectLake - {DATA_LAKEHOUSE_NAME}"

print("✓ Configuration loaded")
print(f"  Workspace ID:        {WORKSPACE_ID}")
print(f"  Data lakehouse:      {DATA_LAKEHOUSE_NAME} ({DATA_LAKEHOUSE_ID})")
print(f"  Metadata lakehouse:  {METADATA_LAKEHOUSE}")
print(f"  DirectLake URL:      {DIRECTLAKE_URL}")
print(f"  Datasource filter:   {DATASOURCE_FILTER or 'all'}")
print(f"  Overwrite existing:  {OVERWRITE}")
print(f"  Fabric token:        obtained ✓")

StatementMeta(, 0e8068c4-3f17-4311-b3a8-b382af87ece3, 7, Finished, Available, Finished, False)

✓ Configuration loaded
  Workspace ID:        a712bac8-d5ad-4773-949e-de8531569016
  Data lakehouse:      h1_ultrastore (6281b20f-5cdc-4f2c-ab2f-87c45866571b)
  Metadata lakehouse:  Metadata_Lakehouse
  DirectLake URL:      https://onelake.dfs.fabric.microsoft.com/a712bac8-d5ad-4773-949e-de8531569016/6281b20f-5cdc-4f2c-ab2f-87c45866571b
  Datasource filter:   all
  Overwrite existing:  True
  Fabric token:        obtained ✓


## Cell 2 — Load Play 3 Metadata

Reads the datasource, field, and lineage inventory from Metadata_Lakehouse.
This drives the entire model generation loop.

In [6]:
def read_metadata_table(table_name):
    """Read a Play 2 metadata table, safely dropping void columns."""
    df = spark.sql(f"SELECT * FROM {METADATA_LAKEHOUSE}.dbo.{table_name}")
    void_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NullType)]
    if void_cols:
        df = df.drop(*void_cols)
    return df.toPandas()

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

df_datasources = read_metadata_table("tableau_datasources")
df_fields      = read_metadata_table("tableau_fields")
df_lineage     = read_metadata_table("tableau_lineage")

# Apply filter
if DATASOURCE_FILTER:
    df_datasources = df_datasources[df_datasources["name"].isin(DATASOURCE_FILTER)]

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

print("✓ Metadata loaded from Play 2")
print(f"  Datasources: {len(df_datasources)}")
print(f"  Fields:      {len(df_fields)}")
print(f"  Lineage:     {len(df_lineage)}")
for _, ds in df_datasources.iterrows():
    ds_id = ds["datasource_id"]
    tables = df_lineage[
        (df_lineage["datasource_id"] == ds_id) &
        (df_lineage["relationship_type"] == "upstream_table")
    ]["related_asset_name"].tolist()
    calc_count = len(df_fields[
        (df_fields["datasource_id"] == ds_id) &
        (df_fields["field_type"] == "CalculatedField")
    ])
    print(f"  • {ds['name']} → tables: {tables}, calculated fields: {calc_count}")


StatementMeta(, 0e8068c4-3f17-4311-b3a8-b382af87ece3, 8, Finished, Available, Finished, False)

✓ Metadata loaded from Play 3
  Datasources: 1
  Fields:      34
  Lineage:     4
  • Superstore Datasource → tables: ['Returns', 'Orders', 'People'], calculated fields: 1


## Cell 3 — TMDL Generators

Functions that generate each TMDL file part for a semantic model.
Based on the TMDL format used by Fabric semantic models (format version 4.2).

**Data type mapping:** Tableau → TMDL
- `STRING` → `string`
- `INTEGER` → `int64`
- `REAL` → `double`
- `BOOLEAN` → `boolean`
- `DATE` → `dateTime`
- `DATETIME` → `dateTime`

**summarizeBy logic:**
- `MEASURE` → `sum`
- `DIMENSION` → `none`
- Unknown/null → `none`

In [ ]:
# ── TYPE MAPPING ──────────────────────────────────────────────────────────────
# Types are driven by the ACTUAL Delta schema (authoritative), NOT Tableau metadata.
# This is the core Play 4 fix: a DirectLake column's dataType must match the physical
# Parquet/Delta column, or the model fails to bind (the prior dateTime-over-varchar bug).
def spark_type_to_tmdl(t):
    """Map a Spark/Delta simpleString type to a TMDL column dataType (or None to skip)."""
    t = (t or "").lower().strip()
    if t.startswith("decimal"):
        return "decimal"
    base = {
        "string": "string", "varchar": "string", "char": "string",
        "byte": "int64", "short": "int64", "integer": "int64", "int": "int64",
        "long": "int64", "bigint": "int64",
        "float": "double", "double": "double",
        "boolean": "boolean",
        "date": "dateTime", "timestamp": "dateTime", "timestamp_ntz": "dateTime",
    }
    if t in base:
        return base[t]
    if t in ("binary", "null", "void") or t.startswith(("array", "map", "struct")):
        return None  # unsupported as a DirectLake model column
    return "string"

def slugify(s):
    s = s.lower().strip()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    return s.strip('_')

def make_delta_table_name(datasource_name, table_name):
    """Match the naming convention used by Play 3."""
    return f"{slugify(datasource_name)}_{slugify(table_name)}"

def clean_col(name):
    for ch in ["(", ")", " ", ",", ";", "{", "}", "/", "\\", "\n", "\t", "="]:
        name = name.replace(ch, "_")
    return name.strip("_")

# ── TMDL identifier quoting ──────────────────────────────────────────────────
# Quote any name with a char outside [A-Za-z0-9_-] or a leading digit (hyphens are
# valid unquoted, e.g. `Sub-Category`). Single-quote and escape embedded quotes.
_UNQUOTED = re.compile(r"^[A-Za-z_][A-Za-z0-9_\-]*$")

def q(name):
    if _UNQUOTED.match(name):
        return name
    return "'" + name.replace("'", "''") + "'"

def _format_string(tmdl_type, summarize):
    if tmdl_type == "dateTime":
        return "Short Date"
    if tmdl_type == "int64":
        return "#,0"
    if tmdl_type in ("double", "decimal") and summarize == "sum":
        return "#,0.00"
    return None

def generate_column_tmdl(col_name, tmdl_type, summarize, is_hidden):
    """One column. col_name is the ACTUAL Delta column name (sourceColumn must match)."""
    lines = [f"\tcolumn {q(col_name)}", f"\t\tdataType: {tmdl_type}"]
    if is_hidden:
        lines.append("\t\tisHidden")
    fmt = _format_string(tmdl_type, summarize)
    if fmt:
        lines.append(f"\t\tformatString: {fmt}")
    lines.append(f"\t\tlineageTag: {uuid.uuid4()}")
    lines.append(f"\t\tsourceLineageTag: {col_name}")
    lines.append(f"\t\tsummarizeBy: {summarize}")
    lines.append(f"\t\tsourceColumn: {col_name}")
    lines.append("")
    lines.append("\t\tannotation SummarizationSetBy = Automatic")
    return "\n" + "\n".join(lines) + "\n"

def generate_table_tmdl(table_display_name, delta_table_name, columns_tmdl, expression_source):
    return (
        f"table {q(table_display_name)}\n"
        f"\tlineageTag: {uuid.uuid4()}\n"
        f"\tsourceLineageTag: [dbo].[{delta_table_name}]\n"
        f"{columns_tmdl}\n"
        f"\tpartition {delta_table_name} = entity\n"
        f"\t\tmode: directLake\n"
        f"\t\tsource\n"
        f"\t\t\tentityName: {delta_table_name}\n"
        f"\t\t\tschemaName: dbo\n"
        f"\t\t\texpressionSource: {q(expression_source)}\n\n"
    )

def tmdl_annotation_value(name, value, indent="\t\t"):
    """Render an `annotation <name> = <value>` line. TMDL reads annotation values
    verbatim to end-of-line, so the formula text is preserved literally (quotes,
    brackets and braces are fine unquoted). Internal line breaks / whitespace runs
    are collapsed to single spaces so the value always stays on one physical line —
    guaranteed-valid TMDL. Translated measures are single-line and round-trip
    byte-for-byte; only multi-line fallback formulas (inert stubs) are normalized."""
    v = " ".join((value or "").split())
    return f"{indent}annotation {name} = {v}\n"

def generate_measure_tmdl(field_name, formula, dax=None):
    """One measure for the _Measures table. When `dax` is provided the measure carries
    the translated DAX expression; otherwise it stays an inert `= 0` stub. EITHER WAY
    the original Tableau formula is ALWAYS preserved as a TableauFormula annotation —
    the unconditional audit/repair safety net for any mistranslation."""
    expr = dax if dax else "0"
    out = (
        f"\n\tmeasure {q(field_name)} = {expr}\n"
        f"\t\tlineageTag: {uuid.uuid4()}\n"
    )
    out += tmdl_annotation_value("TableauFormula", formula)
    if dax:
        out += tmdl_annotation_value("TranslatedBy", "Play4 deterministic translator")
    out += "\t\tannotation SummarizationSetBy = Automatic\n"
    return out

def generate_measures_table_tmdl(measures_tmdl):
    # Canonical measures-holder: a single-row calculated table with one hidden column.
    # The calculated partition (NOT a DirectLake entity) is what made the prior model
    # valid — measure stubs need a home table that doesn't require a Delta binding.
    column = (
        "\n\tcolumn Value\n"
        "\t\tdataType: string\n"
        "\t\tisHidden\n"
        f"\t\tlineageTag: {uuid.uuid4()}\n"
        "\t\tsummarizeBy: none\n"
        "\t\tsourceColumn: [Value]\n"
        "\t\ttype: calculatedTableColumn\n"
    )
    partition = (
        "\tpartition _Measures = calculated\n"
        "\t\tmode: import\n"
        '\t\tsource = Row("Value", BLANK())\n'
    )
    return (
        f"table _Measures\n"
        f"\tlineageTag: {uuid.uuid4()}\n"
        f"{column}"
        f"{measures_tmdl}\n"
        f"{partition}\n"
        f"\tannotation PBI_Id = _Measures\n\n"
    )

def generate_expressions_tmdl(expression_name, directlake_url):
    return (
        f"expression {q(expression_name)} =\n"
        f"\t\tlet\n"
        f'\t\t    Source = AzureStorage.DataLake("{directlake_url}", [HierarchicalNavigation=true])\n'
        f"\t\tin\n"
        f"\t\t    Source\n"
        f"\tlineageTag: {uuid.uuid4()}\n\n"
        f"\tannotation PBI_IncludeFutureArtifacts = False\n\n"
    )

def generate_model_tmdl(table_names, expression_source_name):
    refs = "\n".join([f"ref table {q(t)}" for t in table_names])
    return (
        f"model Model\n"
        f"\tculture: en-US\n"
        f"\tdefaultPowerBIDataSourceVersion: powerBI_V3\n"
        f"\tsourceQueryCulture: en-US\n"
        f"\tdataAccessOptions\n"
        f"\t\tlegacyRedirects\n"
        f"\t\treturnErrorValuesAsNull\n\n"
        f'annotation PBI_QueryOrder = ["{expression_source_name}"]\n\n'
        f"annotation __PBI_TimeIntelligenceEnabled = 0\n\n"
        f'annotation PBI_ProTooling = ["DirectLakeOnOneLakeInWeb","WebModelingEdit"]\n\n'
        f"{refs}\n"
    )

def generate_database_tmdl():
    return "database\n\tcompatibilityLevel: 1604\n"

# ── RELATIONSHIP INFERENCE ───────────────────────────────────────────────────
# Tableau encodes cross-table joins as HIDDEN, disambiguated key fields named
# "<Base> (<Table>)" (e.g. "Region (People)", "Order ID (Returns)"). The matching
# base field "<Base>" lives in the partner table. We pair them, then use the ACTUAL
# landed data to decide which side is unique (the "one" side), so the relationship
# direction (many -> one) is correct regardless of how the join was authored.
_JOINKEY_RE = re.compile(r"^(?P<base>.+) \((?P<tbl>[^()]+)\)$")

def infer_relationships(meta_fields, landed_tables, count_fn):
    """
    meta_fields   : list of dicts with field_name, source_table, field_type, is_hidden
    landed_tables : {table_name: {clean_col: tmdl_type}} actually present in Delta
    count_fn(table_name, clean_col) -> (total, distinct) or None
    Returns list of {from_table, from_col, to_table, to_col, kind}.
    Guards: requires hidden disambiguated key; suffix table must match the key's own
    table; both columns must have landed with COMPATIBLE dtypes; skips self-joins; and
    emits at most ONE relationship per unordered table pair (Fabric allows one active
    path) — extra candidate keys for an already-linked pair are dropped.
    """
    def _s(v):  # normalize pandas NaN / blanks to None
        if v is None or (isinstance(v, float) and v != v):
            return None
        s = str(v).strip()
        return s or None

    def _truthy(v):
        if v is None or (isinstance(v, float) and v != v):
            return False
        if isinstance(v, str):
            return v.strip().lower() in ("true", "1", "yes")
        return bool(v)

    base_index = {}  # non-disambiguated caption -> set(tables exposing it)
    for f in meta_fields:
        if _s(f.get("field_type") or f.get("__typename")) != "ColumnField":
            continue
        nm, st = _s(f.get("field_name") or f.get("name")), _s(f.get("source_table"))
        if not nm or not st or _JOINKEY_RE.match(nm):
            continue
        base_index.setdefault(nm, set()).add(st)

    candidates = []
    for f in meta_fields:
        if _s(f.get("field_type") or f.get("__typename")) != "ColumnField":
            continue
        nm, owner = _s(f.get("field_name") or f.get("name")), _s(f.get("source_table"))
        if not nm or not owner or not _truthy(f.get("is_hidden")):
            continue  # cross-table join keys are always hidden
        m = _JOINKEY_RE.match(nm)
        if not m:
            continue
        base = m.group("base").strip()
        tbl_suffix = _s(m.group("tbl"))
        if tbl_suffix and tbl_suffix.lower() != owner.lower():
            continue  # the "(<Table>)" suffix names the key's own table
        partners = base_index.get(base, set()) - {owner}
        if len(partners) != 1:
            continue  # ambiguous or no partner -> skip
        partner = next(iter(partners))
        if partner == owner:
            continue  # self-join guard
        owner_cols, partner_cols = landed_tables.get(owner, {}), landed_tables.get(partner, {})
        owner_col, base_col = clean_col(nm), clean_col(base)
        if owner_col not in owner_cols or base_col not in partner_cols:
            continue
        if owner_cols.get(owner_col) != partner_cols.get(base_col):
            continue  # dtype mismatch would fail the model deploy
        oc, pc = count_fn(owner, owner_col), count_fn(partner, base_col)
        owner_unique = bool(oc) and oc[0] > 0 and oc[0] == oc[1]
        partner_unique = bool(pc) and pc[0] > 0 and pc[0] == pc[1]
        if owner_unique and not partner_unique:
            frm, frmc, to, toc, kind = partner, base_col, owner, owner_col, "many_to_one"
        elif partner_unique and not owner_unique:
            frm, frmc, to, toc, kind = owner, owner_col, partner, base_col, "many_to_one"
        elif owner_unique and partner_unique:
            frm, frmc, to, toc, kind = partner, base_col, owner, owner_col, "one_to_one"
        else:
            continue  # neither side unique -> many-to-many, skip (avoid a bad model)
        candidates.append({"from_table": frm, "from_col": frmc, "to_table": to,
                           "to_col": toc, "kind": kind})

    # one active relationship per unordered table pair (first wins); drop extras
    rels, used_pairs, seen = [], set(), set()
    for r in candidates:
        key = (r["from_table"], r["from_col"], r["to_table"], r["to_col"])
        pair = frozenset((r["from_table"], r["to_table"]))
        if key in seen or pair in used_pairs:
            continue
        seen.add(key)
        used_pairs.add(pair)
        rels.append(r)
    return rels

def generate_relationships_tmdl(rels):
    """One TMDL relationship per inferred join. Default cardinality is many-to-one,
    which matches from=many -> to=one, so no explicit cardinality props are required."""
    if not rels:
        return None
    blocks = []
    for r in rels:
        blocks.append("\n".join([
            f"relationship {uuid.uuid4()}",
            f"\tfromColumn: {q(r['from_table'])}.{q(r['from_col'])}",
            f"\ttoColumn: {q(r['to_table'])}.{q(r['to_col'])}",
        ]))
    return "\n\n".join(blocks) + "\n"

def generate_pbism():
    return json.dumps({
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json",
        "version": "4.2",
        "settings": {}
    }, indent=2)

def generate_platform(display_name):
    return json.dumps({
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json",
        "metadata": {"type": "SemanticModel", "displayName": display_name},
        "config": {"version": "2.0", "logicalId": "00000000-0000-0000-0000-000000000000"}
    }, indent=2)

def encode(text):
    return base64.b64encode(text.encode('utf-8')).decode('utf-8')

print("✓ TMDL generators ready (types driven by actual Delta schema)")


## Cell 3b — Tableau-calc → DAX Translator

Deterministically translates the *simple* subset of Tableau calculated fields (single-field aggregations + arithmetic) into working DAX measures. Anything more complex falls back to an inert `= 0` stub. The original Tableau formula is always preserved as a `TableauFormula` annotation so a mistranslation can be audited and repaired by hand.

In [ ]:
# ── Tableau-calc → DAX measure translator (deterministic, no LLM) ─────────────
# Translates a SAFE subset of Tableau calculated fields into working DAX measures:
#   • aggregations over a single bare field: SUM, AVG, MIN, MAX, COUNT, COUNTD, MEDIAN
#   • arithmetic between those terms / numeric literals: + - * /, parentheses, unary minus
# Anything outside this subset (IF/CASE, LOD {FIXED/…}, string/date/window funcs,
# nested arithmetic inside an aggregation, references to other calcs, unresolved or
# ambiguous fields, cross-table arithmetic) deterministically FALLS BACK to an inert
# `= 0` stub. The original Tableau formula is preserved as an annotation either way.
_AGG_MAP = {
    "SUM": "SUM", "AVG": "AVERAGE", "MIN": "MIN", "MAX": "MAX",
    "MEDIAN": "MEDIAN", "COUNT": "COUNTA", "COUNTD": "DISTINCTCOUNTNOBLANK",
}
# COUNT  → COUNTA               (Tableau COUNT = non-null of ANY type; DAX COUNT errors on text)
# COUNTD → DISTINCTCOUNTNOBLANK (plain DISTINCTCOUNT counts BLANK → off-by-one vs Tableau)
_NUMERIC_TYPES = {"int64", "double", "decimal"}


class _CalcError(Exception):
    """Raised on any construct outside the supported subset → caller falls back."""


def _dax_table(name):
    # DAX table reference: single-quoted, embedded single quotes doubled.
    return "'" + name.replace("'", "''") + "'"


def _dax_col(name):
    # DAX column reference: [bracketed], embedded ] doubled.
    return "[" + name.replace("]", "]]") + "]"


def _norm_number(tok):
    # .5 → 0.5 ; 1. → 1.0 (DAX dislikes a bare leading/trailing dot)
    if tok.startswith("."):
        tok = "0" + tok
    if tok.endswith("."):
        tok = tok + "0"
    return tok


_NUM_RE = re.compile(r"\d+\.?\d*|\.\d+")
_ID_RE = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")


def _tokenize(formula):
    s = formula or ""
    i, n = 0, len(s)
    toks = []
    while i < n:
        c = s[i]
        if c in " \t\r\n":
            i += 1
            continue
        if c == "[":
            j = s.find("]", i + 1)
            if j == -1:
                raise _CalcError("unterminated field reference")
            toks.append(("field", s[i + 1:j]))
            i = j + 1
            continue
        if c in "+-*/()":
            toks.append(("op", c))
            i += 1
            continue
        m = _NUM_RE.match(s, i)
        if m and (c.isdigit() or c == "."):
            toks.append(("num", m.group(0)))
            i = m.end()
            continue
        m = _ID_RE.match(s, i)
        if m:
            toks.append(("id", m.group(0)))
            i = m.end()
            continue
        raise _CalcError(f"unsupported character {c!r}")
    return toks


# Recursive-descent parser with correct DAX/arithmetic precedence:
#   expr := add ; add := mul (('+'|'-') mul)* ; mul := unary (('*'|'/') unary)*
#   unary := '-' unary | primary ; primary := agg | number | '(' expr ')'
#   agg := AGGFUNC '(' '[' fieldref ']' ')'
class _Parser:
    def __init__(self, toks, resolver, tables_used):
        self.toks = toks
        self.pos = 0
        self.resolver = resolver
        self.tables_used = tables_used

    def _peek(self):
        return self.toks[self.pos] if self.pos < len(self.toks) else (None, None)

    def _next(self):
        t = self._peek()
        self.pos += 1
        return t

    def _expect_op(self, ch):
        k, v = self._peek()
        if k != "op" or v != ch:
            raise _CalcError(f"expected {ch!r}")
        self.pos += 1

    def parse(self):
        node = self._add()
        if self.pos != len(self.toks):
            raise _CalcError("unexpected trailing tokens")
        return node

    def _add(self):
        left = self._mul()
        while self._peek() == ("op", "+") or self._peek() == ("op", "-"):
            op = self._next()[1]
            right = self._mul()
            left = f"{left} {op} {right}"
        return left

    def _mul(self):
        left = self._unary()
        while self._peek() == ("op", "*") or self._peek() == ("op", "/"):
            op = self._next()[1]
            right = self._unary()
            left = f"DIVIDE({left}, {right})" if op == "/" else f"{left} * {right}"
        return left

    def _unary(self):
        if self._peek() == ("op", "-"):
            self._next()
            operand = self._unary()
            return f"-({operand})"  # parenthesize so '--' never forms a DAX comment
        return self._primary()

    def _primary(self):
        k, v = self._peek()
        if k == "id":
            return self._agg()
        if k == "num":
            self._next()
            return _norm_number(v)
        if k == "op" and v == "(":
            self._next()
            inner = self._add()
            self._expect_op(")")
            return f"({inner})"
        raise _CalcError("expected aggregation, number, or '('")

    def _agg(self):
        name = self._next()[1].upper()
        if name not in _AGG_MAP:
            raise _CalcError(f"unsupported function {name}")
        self._expect_op("(")
        k, v = self._peek()
        if k != "field":
            raise _CalcError(f"{name} argument must be a single bare [field]")
        self._next()
        self._expect_op(")")
        resolved = self.resolver(v)
        if resolved is None:
            raise _CalcError(f"unresolved/ambiguous field [{v}]")
        table, col, tmdl_type = resolved
        # Reject aggregates invalid for the column's data type (would emit DAX that errors).
        if name in ("SUM", "AVG", "MEDIAN") and tmdl_type not in _NUMERIC_TYPES:
            raise _CalcError(f"{name} requires a numeric field, got {tmdl_type} for [{v}]")
        if name in ("MIN", "MAX") and tmdl_type not in (_NUMERIC_TYPES | {"dateTime"}):
            raise _CalcError(f"{name} requires a numeric/date field, got {tmdl_type} for [{v}]")
        self.tables_used.add(table)
        return f"{_AGG_MAP[name]}({_dax_table(table)}{_dax_col(col)})"


def translate_tableau_calc_to_dax(formula, resolver):
    """Translate a SIMPLE Tableau calc to DAX. Returns (dax|None, reason, tables_used).

    dax is None on any unsupported construct → caller keeps the inert `= 0` stub.
    resolver(caption) → (table_display_name, clean_col, tmdl_type) | None.
    """
    tables_used = set()
    f = (formula or "").strip()
    if not f:
        return None, "empty formula", tables_used
    try:
        toks = _tokenize(f)
        if not toks:
            return None, "empty formula", tables_used
        dax = _Parser(toks, resolver, tables_used).parse()
        # Single-table only: terms spanning >1 table fall back (a relationship path
        # does not guarantee the DAX filter context reproduces Tableau's result).
        if len(tables_used) > 1:
            return None, "cross-table arithmetic (terms span multiple tables)", tables_used
        return dax, "ok", tables_used
    except _CalcError as e:
        return None, str(e), tables_used


print("✓ Tableau-calc → DAX translator ready (deterministic, single-table aggregations)")


### Cell 3c — Translator self-test

Runs offline (pure Python, no Spark) so the translator is verified every run. Raises if any case regresses.

In [ ]:
# ── Self-test: translator + measure rendering (pure Python, no Spark) ─────────
# Runs offline so the deterministic translator is verified every time the notebook
# executes, independent of any live datasource. Raises if any case regresses.
def _run_translator_self_tests():
    fields = {
        "Profit": ("Orders", "Profit", "decimal"),
        "Sales": ("Orders", "Sales", "decimal"),
        "Quantity": ("Orders", "Quantity", "int64"),
        "Order Date": ("Orders", "Order_Date", "dateTime"),
        "Region": ("Orders", "Region", "string"),
        "People Count": ("People", "People_Count", "int64"),
    }
    resolver = lambda cap: fields.get(cap)
    tx = lambda f: translate_tableau_calc_to_dax(f, resolver)[0]

    ok = [
        ("SUM([Profit])/SUM([Sales])", "DIVIDE(SUM('Orders'[Profit]), SUM('Orders'[Sales]))"),
        ("SUM([Sales])", "SUM('Orders'[Sales])"),
        ("AVG([Sales])", "AVERAGE('Orders'[Sales])"),
        ("MIN([Sales])", "MIN('Orders'[Sales])"),
        ("MAX([Sales])", "MAX('Orders'[Sales])"),
        ("MEDIAN([Sales])", "MEDIAN('Orders'[Sales])"),
        ("COUNT([Sales])", "COUNTA('Orders'[Sales])"),
        ("COUNTD([Region])", "DISTINCTCOUNTNOBLANK('Orders'[Region])"),
        ("MIN([Order Date])", "MIN('Orders'[Order_Date])"),
        ("SUM([Sales])+SUM([Profit])", "SUM('Orders'[Sales]) + SUM('Orders'[Profit])"),
        ("SUM([Sales])-SUM([Profit])", "SUM('Orders'[Sales]) - SUM('Orders'[Profit])"),
        ("SUM([Sales])*SUM([Profit])", "SUM('Orders'[Sales]) * SUM('Orders'[Profit])"),
        ("SUM([Profit])+SUM([Sales])*SUM([Quantity])",
         "SUM('Orders'[Profit]) + SUM('Orders'[Sales]) * SUM('Orders'[Quantity])"),
        ("(SUM([Profit])+SUM([Sales]))*SUM([Quantity])",
         "(SUM('Orders'[Profit]) + SUM('Orders'[Sales])) * SUM('Orders'[Quantity])"),
        ("SUM([Sales])/SUM([Profit])/SUM([Quantity])",
         "DIVIDE(DIVIDE(SUM('Orders'[Sales]), SUM('Orders'[Profit])), SUM('Orders'[Quantity]))"),
        ("SUM([Profit])/SUM([Sales])*100",
         "DIVIDE(SUM('Orders'[Profit]), SUM('Orders'[Sales])) * 100"),
        ("SUM([Sales])*.5", "SUM('Orders'[Sales]) * 0.5"),
        ("-SUM([Profit])", "-(SUM('Orders'[Profit]))"),
        ("SUM([Sales]) - -SUM([Profit])", "SUM('Orders'[Sales]) - -(SUM('Orders'[Profit]))"),
    ]
    # Everything below must FALL BACK (translator returns None).
    fallbacks = [
        'IF [Sales]>0 THEN "y" ELSE "n" END',
        "{FIXED [Region] : SUM([Sales])}",
        "ZN(SUM([Sales]))",
        "SUM([Sales]-[Profit])", "SUM([Sales]+1)", "SUM(-[Sales])",
        "[Sales]+[Profit]", "SUM([Nonexistent])", "SUM(5)", "",
        "LEFT([Region],3)", "SUM([Sales]) SUM([Profit])",
        "SUM([Sales])/SUM([People Count])",          # cross-table
        "SUM([Region])", "AVG([Order Date])", "MEDIAN([Region])",  # type-invalid
        "SUM([Sales]) + WINDOW_SUM(SUM([Profit]))",
        "IF SUM([Sales]) > 0 THEN SUM([Profit]) END",
    ]
    failures = []
    for f, want in ok:
        got = tx(f)
        if got != want:
            failures.append(f"translate {f!r}: got {got!r}, want {want!r}")
    for f in fallbacks:
        got = tx(f)
        if got is not None:
            failures.append(f"expected fallback for {f!r}, got DAX {got!r}")

    # Measure rendering: translated keeps formula + TranslatedBy; stub keeps formula only.
    m_tr = generate_measure_tmdl("Profit Ratio", "SUM([Profit])/SUM([Sales])",
                                 "DIVIDE(SUM('Orders'[Profit]), SUM('Orders'[Sales]))")
    if "= DIVIDE(SUM('Orders'[Profit]), SUM('Orders'[Sales]))" not in m_tr:
        failures.append("translated measure missing DAX expression")
    if "annotation TableauFormula = SUM([Profit])/SUM([Sales])" not in m_tr:
        failures.append("translated measure missing original-formula annotation")
    if "annotation TranslatedBy" not in m_tr:
        failures.append("translated measure missing TranslatedBy annotation")
    m_stub = generate_measure_tmdl("Complex", "IF [x]>0\nTHEN 1\nEND", None)
    if "= 0" not in m_stub:
        failures.append("stub measure should be = 0")
    if "annotation TableauFormula = IF [x]>0 THEN 1 END" not in m_stub:
        failures.append("stub measure should preserve formula on one line")
    if "annotation TranslatedBy" in m_stub:
        failures.append("stub measure must NOT claim TranslatedBy")

    if failures:
        for f in failures:
            print("  ✗", f)
        raise AssertionError(f"{len(failures)} translator self-test(s) failed")
    print(f"✓ Translator self-tests passed ({len(ok)} translations, {len(fallbacks)} fallbacks, 2 render checks)")


_run_translator_self_tests()


## Cell 4 — Main Loop: Generate and Deploy Semantic Models

For each datasource:
1. Reads upstream tables from `tableau_lineage`
2. Reads fields from `tableau_fields` — columns go to their source tables, calculated fields go to `_Measures`
3. Generates TMDL definition parts
4. Deploys via Fabric REST API `createItem` endpoint
5. Polls for completion if async

> **Relationships:** cross-table relationships are inferred from Tableau's hidden join
> keys and emitted as `definition/relationships.tmdl` (many→one direction from the data).

> **On failure:** each model is deployed independently. If one fails the loop continues.

In [ ]:
def _as_bool(v):
    """Robust truthiness for an is_hidden value that may be bool, int, str, or NA."""
    if v is None:
        return False
    if isinstance(v, float) and v != v:  # NaN
        return False
    if isinstance(v, str):
        return v.strip().lower() in ("true", "1", "yes")
    return bool(v)

def get_existing_models():
    """Get dict of existing semantic model display names → IDs in workspace."""
    resp = requests.get(
        f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/semanticModels",
        headers=HEADERS
    )
    resp.raise_for_status()
    return {m["displayName"]: m["id"] for m in resp.json().get("value", [])}

def deploy_semantic_model(display_name, parts, existing_id=None):
    """Deploy via Fabric REST API. Updates existing if OVERWRITE, else creates. Polls async."""
    definition = {"format": "TMDL", "parts": parts}

    if existing_id and OVERWRITE:
        resp = requests.post(
            f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/semanticModels/{existing_id}/updateDefinition",
            headers=HEADERS, json={"definition": definition})
    else:
        resp = requests.post(
            f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items",
            headers=HEADERS,
            json={"displayName": display_name, "type": "SemanticModel", "definition": definition})

    if resp.status_code == 202:
        operation_url = resp.headers.get("Location")
        retry_after = int(resp.headers.get("Retry-After", 20))
        # Poll until the async operation leaves the Running state.
        for _ in range(30):
            time.sleep(retry_after)
            poll = requests.get(operation_url, headers=HEADERS)
            poll.raise_for_status()
            result = poll.json()
            status = result.get("status")
            if status == "Succeeded":
                return result
            if status in ("Failed", "Undefined"):
                raise Exception(f"Async operation failed: {result}")
            retry_after = int(poll.headers.get("Retry-After", retry_after))
        raise Exception("Async operation did not complete in time")

    resp.raise_for_status()
    return resp.json()

def get_delta_schema(delta_table):
    """Return [(column_name, spark_type_simpleString), ...] for a landed Delta table,
    or None if the table genuinely does not exist. Real errors (permissions, bad
    namespace) are re-raised so they aren't silently misread as a missing table."""
    full = f"{DATA_LAKEHOUSE_NAME}.dbo.{delta_table}"
    try:
        fields = spark.table(full).schema.fields
    except Exception as e:
        msg = str(e).upper()
        if ("TABLE_OR_VIEW_NOT_FOUND" in msg or "NOT FOUND" in msg
                or "CANNOT BE FOUND" in msg or "DOES NOT EXIST" in msg
                or "PATH DOES NOT EXIST" in msg):
            return None
        raise
    return [(f.name, f.dataType.simpleString()) for f in fields]

def build_model_parts(ds_name, ds_id):
    """
    Build all TMDL parts for a datasource. Columns and types come from the ACTUAL
    landed Delta schema (Play 3 output); summarizeBy/isHidden are looked up from Play 2
    metadata by sanitized name. Tables that never landed are skipped (no broken bindings).
    """
    upstream_tables = df_lineage[
        (df_lineage["datasource_id"] == ds_id) &
        (df_lineage["relationship_type"] == "upstream_table")
    ]["related_asset_name"].tolist()

    ds_fields = df_fields[df_fields["datasource_id"] == ds_id]

    parts = []
    table_names = []
    skipped_tables = []
    landed_cols = {}  # table_name -> set(actual Delta column names) for relationship inference

    # ── One table per upstream source — built from the real Delta schema ──────
    for table_name in upstream_tables:
        delta_table = make_delta_table_name(ds_name, table_name)
        schema = get_delta_schema(delta_table)
        if schema is None:
            skipped_tables.append(table_name)
            print(f"    ⚠ Delta table not found, skipping: {delta_table}")
            continue
        landed_cols[table_name] = {c: spark_type_to_tmdl(t) for c, t in schema}

        # Metadata lookup keyed by sanitized name: clean_col(field) -> (role, is_hidden)
        meta_by_col = {}
        table_meta = ds_fields[
            (ds_fields["field_type"] == "ColumnField") &
            (ds_fields["source_table"] == table_name)
        ]
        for _, f in table_meta.iterrows():
            meta_by_col[clean_col(f["field_name"])] = (
                f.get("role", "DIMENSION"), _as_bool(f.get("is_hidden", False)))

        columns_tmdl = ""
        col_count = 0
        for col_name, spark_type in schema:
            tmdl_type = spark_type_to_tmdl(spark_type)
            if tmdl_type is None:
                print(f"    ⚠ Unsupported type {spark_type} on {delta_table}.{col_name}, skipping column")
                continue
            role, is_hidden = meta_by_col.get(col_name, ("DIMENSION", False))
            summarize = "sum" if (role == "MEASURE" and tmdl_type in ("int64", "double", "decimal")) else "none"
            columns_tmdl += generate_column_tmdl(col_name, tmdl_type, summarize, is_hidden)
            col_count += 1

        if col_count == 0:
            skipped_tables.append(table_name)
            print(f"    ⚠ No supported columns for {delta_table}, skipping table")
            continue

        table_tmdl = generate_table_tmdl(table_name, delta_table, columns_tmdl, EXPRESSION_SOURCE_NAME)
        parts.append({
            "path": f"definition/tables/{delta_table}.tmdl",
            "payload": encode(table_tmdl),
            "payloadType": "InlineBase64"
        })
        table_names.append(table_name)

    if not table_names:
        raise Exception(f"No Delta tables landed for '{ds_name}' — run Play 3 first")

    # ── Relationships — inferred from hidden disambiguated join keys ──────────
    # Cardinality comes from the ACTUAL landed data so the many->one direction is
    # correct. Emitted as definition/relationships.tmdl (auto-discovered by Fabric).
    from pyspark.sql import functions as F
    def _cardinality(tbl, col):
        dt = make_delta_table_name(ds_name, tbl)
        try:
            row = spark.table(f"{DATA_LAKEHOUSE_NAME}.dbo.{dt}").agg(
                F.count(F.lit(1)).alias("t"),
                F.countDistinct(F.col(col)).alias("d")).collect()[0]
            return (int(row["t"]), int(row["d"]))
        except Exception:
            return None

    rels = infer_relationships(ds_fields.to_dict("records"), landed_cols, _cardinality)
    rels_tmdl = generate_relationships_tmdl(rels)
    if rels_tmdl:
        parts.append({
            "path": "definition/relationships.tmdl",
            "payload": encode(rels_tmdl),
            "payloadType": "InlineBase64"
        })
        print(f"    Relationships inferred: {len(rels)}")
        for r in rels:
            print(f"      {r['from_table']}.{r['from_col']} → {r['to_table']}.{r['to_col']} ({r['kind']})")
    else:
        print("    Relationships inferred: 0 (no cross-table join keys detected)")

    # ── Field resolver for DAX measure translation ────────────────────────────
    # caption → (table, clean_col, tmdl_type), resolved ONLY when unambiguous: exactly
    # one EMITTED table exposes a ColumnField with that caption whose sanitized column
    # actually landed and is not a clean-name collision. Anything else → no resolution
    # (the calc using it falls back to a stub), so we never bind a measure to the wrong column.
    emitted = set(table_names)
    _cap_to_col = {}    # (table, caption) → clean_col
    _col_captions = {}  # (table, clean_col) → set(captions)  (clean-name collision detector)
    for _, fr in ds_fields[ds_fields["field_type"] == "ColumnField"].iterrows():
        st = fr.get("source_table")
        cap = fr.get("field_name")
        if not st or cap is None or st not in emitted:
            continue
        cc = clean_col(cap)
        if landed_cols.get(st, {}).get(cc) is None:
            continue  # column never landed, or was an unsupported type (skipped)
        _cap_to_col[(st, cap)] = cc
        _col_captions.setdefault((st, cc), set()).add(cap)

    def resolve_field(caption):
        hits = []
        for st in emitted:
            cc = _cap_to_col.get((st, caption))
            if cc is None:
                continue
            if len(_col_captions.get((st, cc), ())) != 1:
                continue  # two captions sanitize to the same column here → ambiguous
            hits.append((st, cc, landed_cols[st][cc]))
        return hits[0] if len(hits) == 1 else None

    # ── _Measures (only when calculated fields exist) ─────────────────────────
    # Each Tableau calc is translated to DAX when it falls in the supported subset;
    # otherwise it stays an inert `= 0` stub. The original formula is preserved either way.
    calc_fields = ds_fields[ds_fields["field_type"] == "CalculatedField"]
    measure_report = []
    if len(calc_fields) > 0:
        measures_tmdl = ""
        for _, field in calc_fields.iterrows():
            fname = field["field_name"]
            formula = field.get("formula", "")
            dax, reason, _ = translate_tableau_calc_to_dax(formula, resolve_field)
            measures_tmdl += generate_measure_tmdl(fname, formula, dax)
            measure_report.append({
                "datasource": ds_name, "measure": fname,
                "status": "translated" if dax else "stub",
                "dax": dax or "", "reason": reason, "formula": formula,
            })
        parts.append({
            "path": "definition/tables/_Measures.tmdl",
            "payload": encode(generate_measures_table_tmdl(measures_tmdl)),
            "payloadType": "InlineBase64"
        })
        table_names.append("_Measures")

    # ── Expressions (DirectLake connection) ──────────────────────────────────
    parts.append({
        "path": "definition/expressions.tmdl",
        "payload": encode(generate_expressions_tmdl(EXPRESSION_SOURCE_NAME, DIRECTLAKE_URL)),
        "payloadType": "InlineBase64"
    })
    # ── Model root ───────────────────────────────────────────────────────────
    parts.append({
        "path": "definition/model.tmdl",
        "payload": encode(generate_model_tmdl(table_names, EXPRESSION_SOURCE_NAME)),
        "payloadType": "InlineBase64"
    })
    # ── Database ─────────────────────────────────────────────────────────────
    parts.append({
        "path": "definition/database.tmdl",
        "payload": encode(generate_database_tmdl()),
        "payloadType": "InlineBase64"
    })
    # ── definition.pbism ─────────────────────────────────────────────────────
    parts.append({
        "path": "definition.pbism",
        "payload": encode(generate_pbism()),
        "payloadType": "InlineBase64"
    })
    # ── .platform ────────────────────────────────────────────────────────────
    parts.append({
        "path": ".platform",
        "payload": encode(generate_platform(ds_name)),
        "payloadType": "InlineBase64"
    })

    return parts, table_names, skipped_tables, measure_report

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
REFRESH_AFTER_DEPLOY = True  # Frame DirectLake so the model is immediately query-ready

def refresh_model(model_id):
    """Trigger + await a full refresh so DirectLake frames against the Delta tables.
    A freshly-deployed DirectLake model returns 'table not refreshed' until this runs."""
    url = f"https://api.powerbi.com/v1.0/myorg/datasets/{model_id}/refreshes"
    resp = requests.post(url, headers=HEADERS, json={"type": "full", "commitMode": "transactional"})
    if resp.status_code not in (200, 202):
        raise Exception(f"refresh POST {resp.status_code}: {resp.text[:200]}")
    for _ in range(30):
        time.sleep(8)
        top = requests.get(url + "?$top=1", headers=HEADERS).json().get("value", [])
        if not top:
            continue
        status = top[0].get("status")
        if status == "Completed":
            return
        if status == "Failed":
            raise Exception(f"refresh failed: {json.dumps(top[0])[:300]}")
    raise Exception("refresh did not complete in time")

results = []
errors  = []
all_measure_reports = []

print(f"Starting semantic model generation — {datetime.utcnow().isoformat()}")
print("=" * 60)

existing_models = get_existing_models()
print(f"  Existing models in workspace: {list(existing_models.keys())}")

for _, ds_row in df_datasources.iterrows():
    ds_id   = ds_row["datasource_id"]
    ds_name = ds_row["name"]
    model_display_name = f"{ds_name} — Fabric Semantic Model"

    print(f"\n── {ds_name} ──")

    existing_id = existing_models.get(model_display_name)
    if existing_id and not OVERWRITE:
        print(f"  ⚠ Skipped — model already exists and OVERWRITE=False")
        continue

    try:
        parts, table_names, skipped, mreport = build_model_parts(ds_name, ds_id)
        print(f"  Tables: {table_names}")
        if skipped:
            print(f"  Skipped (not landed): {skipped}")
        print(f"  Parts:  {len(parts)} TMDL files generated")
        if mreport:
            n_tr = sum(1 for _m in mreport if _m["status"] == "translated")
            print(f"  Measures: {n_tr} translated to DAX, {len(mreport) - n_tr} left as annotated stub")
            for _m in mreport:
                if _m["status"] == "translated":
                    print(f"      \u2713 {_m['measure']}  \u2192  {_m['dax']}")
                else:
                    print(f"      \u2022 {_m['measure']}  (stub \u2014 {_m['reason']})")
            all_measure_reports.extend(mreport)

        result = deploy_semantic_model(model_display_name, parts, existing_id)
        action = "updated" if existing_id else "created"
        print(f"  ✓ Semantic model {action}: '{model_display_name}'")

        if REFRESH_AFTER_DEPLOY:
            model_id = existing_id or get_existing_models().get(model_display_name)
            if not model_id:
                print(f"  ⚠ Deployed, but new model id not yet listed — refresh manually in Fabric")
            else:
                try:
                    refresh_model(model_id)
                    print(f"  ✓ DirectLake refreshed — model is query-ready")
                except Exception as re:
                    print(f"  ⚠ Deployed, but refresh failed (refresh manually in Fabric): {re}")

        results.append({"datasource": ds_name, "model": model_display_name,
                        "tables": table_names, "skipped": skipped, "action": action})

    except Exception as e:
        print(f"  ✗ Failed: {e}")
        errors.append({"datasource": ds_name, "error": str(e)})

print(f"\n{'=' * 60}")
print(f"✓ Complete — {datetime.utcnow().isoformat()}")
print(f"  Models deployed: {len(results)}")
print(f"  Errors:          {len(errors)}")
if errors:
    for e in errors:
        print(f"  ✗ {e['datasource']}: {e['error']}")

# ── Measure translation summary (audit: which calcs still need a manual DAX rewrite) ──
if all_measure_reports:
    _tr = [m for m in all_measure_reports if m["status"] == "translated"]
    _st = [m for m in all_measure_reports if m["status"] == "stub"]
    print(f"\n{'=' * 60}")
    print(f"MEASURE TRANSLATION: {len(_tr)} translated to DAX, {len(_st)} left as stub")
    if _st:
        print("  Needs manual review (inert `= 0`; original Tableau formula kept as annotation):")
        for m in _st:
            print(f"    \u2022 [{m['datasource']}] {m['measure']}: {m['reason']}")
            print(f"        {m['formula']}")


## Cell 5 — Verification

Confirms deployed models and lists them with their tables.

In [9]:
print("=" * 60)
print("VERIFICATION")
print("=" * 60)

# Re-fetch models from workspace
resp = requests.get(
    f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/semanticModels",
    headers=HEADERS
)
resp.raise_for_status()
all_models = resp.json().get("value", [])

if results:
    print("\n── Models deployed this run ──")
    for r in results:
        print(f"  ✓ [{r['action']}] {r['model']}")
        print(f"         Tables: {r['tables']}")

print(f"\n── All semantic models in workspace ──")
for m in all_models:
    marker = "← new" if m["displayName"] in [r["model"] for r in results] else ""
    print(f"  • {m['displayName']} {marker}")

print(f"\n  ✓ Semantic models ready in Fabric workspace")
print(f"  ✓ DirectLake → {DATA_LAKEHOUSE_NAME}")
print(f"  ✓ Relationships auto-mapped from Tableau join keys")


StatementMeta(, 0e8068c4-3f17-4311-b3a8-b382af87ece3, 11, Finished, Available, Finished, False)

VERIFICATION

── All semantic models in workspace ──
  • Test Model 
  • Meta_data_semantic_Model 
  • Test Bim 

  ✓ Semantic models ready in Fabric workspace
  ✓ DirectLake → h1_ultrastore
  ✓ Add relationships in Fabric to complete the model
